# 06 · PrimateAI — the semi-supervised alternative

**PrimateAI** (Sundaram et al. 2018, *Nat Genet*, PMID 30038395) is a deep net that dodges the need for curated pathogenic labels: it trains on a **proxy for benignity** — missense variants common in humans or seen in other primates. That makes it **semi-supervised**, with only **medium** circularity vs ClinVar.

> ✅ **REAL DATA (subset).** PrimateAI for CFTR from **dbNSFP v5.0a** — `data/primateai_cftr.csv`, `build_primateai.py`. ⚠️ **Coverage:** dbNSFP's ClinVar-re-annotated subset, so **~1,976 observed** CFTR variants (NOT saturation). Non-commercial. `source == 'REAL'`.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 5 · PrimateAI — the *semi-supervised* alternative

**PrimateAI** (Sundaram et al. 2018, *Nature Genetics*, **PMID 30038395**) is a **deep neural
network**, but it dodges the "we need labelled pathogenic variants" problem with a neat trick.

Instead of training on curated *pathogenic* labels, it trains on a **proxy for benignity**:

> A missense variant that is **common** in healthy humans — or is seen as the normal amino acid in
> **other primates** (chimp, gorilla, …) — is very likely **tolerated / benign**. Evolution has
> already "tested" it.

So PrimateAI learns from **hundreds of thousands of common human & non-human-primate missense
variants** as its benign class. This is why it's called **semi-supervised**: it uses labels, but
those labels are a *population-frequency proxy*, **not** clinical pathogenic/benign assertions.

| Property | PrimateAI |
|---|---|
| Learning type | **Semi-supervised** (benign proxy from primate/common variants) |
| Score range | **0 → 1** (higher = more likely pathogenic) |
| Pathogenic cut | **≥ 0.803** |
| Circularity vs ClinVar | **Medium** — it never trained on ClinVar *pathogenic* labels, but common-variant proxies can still overlap benign ClinVar entries |


In [2]:
tk.THRESHOLDS['primate_ai']

{'path': 0.803, 'benign': 0.483}

## 2 · Load PrimateAI and make a call

Score 0-1, higher = worse, pathogenic cut `>= 0.803`.

In [3]:
primateai = tk.load_primateai()   # REAL — dbNSFP subset (~1,976 observed CFTR variants)
print(f"{len(primateai):,} REAL PrimateAI variants | source: {primateai['source'].unique().tolist()}")
print('score range:', primateai['primate_ai_score'].min(), '->', primateai['primate_ai_score'].max(),
      '| pathogenic (>= 0.803):', int((primateai['primate_ai_score'] >= 0.803).sum()))
primateai['pai_call'] = primateai['primate_ai_score'].apply(lambda s: tk.call_from_score(s, 'primate_ai'))
primateai[['protein_variant', 'primate_ai_score', 'pai_call', 'source']].head(10)

1,976 REAL PrimateAI variants | source: ['REAL']
score range: 0.176448047161 -> 0.932814955711 | pathogenic (>= 0.803): 132


,protein_variant,primate_ai_score,pai_call,source
0,Q2P,0.612750,uncertain,REAL
1,R3W,0.439323,benign,REAL
2,R3M,0.399057,benign,REAL
3,S4L,0.585684,uncertain,REAL
4,P5S,0.569334,uncertain,REAL
5,P5R,0.643677,uncertain,REAL
6,P5L,0.645004,uncertain,REAL
7,L6V,0.522222,uncertain,REAL
8,K8R,0.525494,uncertain,REAL
9,A9V,0.517085,uncertain,REAL


## 3 · How to get the REAL PrimateAI scores

PrimateAI / PrimateAI-3D scores are distributed by **Illumina** (accept a data-use agreement, then download the precomputed files). Keyed by **genomic coordinate** — join on `(chr, pos, ref, alt)`, minding CFTR's minus strand.

In [4]:
info = tk.TOOL_REGISTRY['PrimateAI']
for key, val in info.items():
    print(f'  {key:12s}: {val}')

  kind        : missense
  learning    : semi-supervised
  signal      : deep net trained on common human/primate missense as a benign proxy
  circularity : medium
  pmid        : 30038395


## The shared A1 worked-example panel — **PrimateAI** in context

The same ~14 famous CFTR **missense** variants are scored by *every* A1 tool
(notebooks 01–08), so you can follow one fixed panel across the whole series. Your
column here is **`PAI`**; the CFTR2 / ClinVar truth columns are shown for context.

`tk.a1_panel()` is the single source of truth (defined in `toolkit.py`); it uses the
REAL extracts where present (missing extracts → NaN).

In [5]:
tk.a1_panel()

,protein_variant,gnomad_af,AM,EVE,ESM1b,REVEL,PAI,cftr2_class,clinvar_call
0,G551D,0.000404,0.9897,0.939098,-12.123,0.990,0.757039,CF-causing,pathogenic
1,F508del,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,R117H,0.002199,0.2985,0.819590,-5.659,0.807,0.538583,Varying clinical consequence,pathogenic
3,R334W,0.000092,0.6563,0.950636,-8.179,0.816,0.715262,CF-causing,pathogenic
4,G85E,0.000065,0.9881,0.922779,-11.889,0.940,0.775947,CF-causing,pathogenic
5,D1152H,0.000407,0.8597,0.842551,-9.784,0.657,0.687562,Varying clinical consequence,uncertain
6,R668C,0.008450,0.1040,0.921812,-9.968,0.706,0.713809,Non CF-causing,uncertain
7,Y161C,0.000002,0.8006,0.934065,-8.857,0.969,0.844116,No interpretation available,uncertain
8,G970D,0.000007,0.7638,0.930015,-12.000,0.985,0.621215,CF-causing,pathogenic
9,S912L,0.000663,0.1245,0.085494,-3.418,0.543,0.301304,Varying clinical consequence,uncertain


## Key takeaways

1. **PrimateAI** is **semi-supervised** (benignity learned from common human/primate variants) → **medium** circularity. Cut `>= 0.803`.
2. Now **REAL** — but a **dbNSFP ClinVar subset** (~1,976 observed CFTR variants), not saturation. Honestly labelled; covers ~half of A1's observed set.
3. Non-commercial (dbNSFP CC BY-NC-ND) — raw parquet kept external.

**Next:** notebook 07 — **ClinVar**.